In [7]:
import os
import json

import numpy as np
import pandas as pd

In [8]:
# runs = [
#     os.path.join('experiments', str(x)) for x in range(744, 824)
# ] # without +/- fix in Z_0 calculation

runs = [
    os.path.join('experiments', str(x)) for x in range(1154, 1194)
] # fixed +/- in Z_0 calculation

# assert len(runs) == 80, len(runs)

# assert len(runs) == 40, len(runs)

In [9]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [10]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error',
        'time_to_1p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))
    # assert info['use_cpu'] == False
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item()
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] == min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item()
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item()

    try:
        first_1p_epoch = int(log[log['rel_l2_error'] < 0.01].iloc[0]['epoch'])
        info['time_to_1p_rel_l2_error'] = log[log['epoch'] <= first_1p_epoch]['training_time'].sum().item()
    except:
        print(run, log['rel_l2_error'].min())
        info['time_to_1p_rel_l2_error'] = float('inf')
    
    df.loc[len(df)] = info

experiments/1166 0.0316368210525805
experiments/1168 0.0378556459270344
experiments/1170 0.0344043507839562
experiments/1172 0.0390367439023696
experiments/1174 0.0365374977951806
experiments/1180 0.0111962913528385
experiments/1186 0.026370988074047
experiments/1188 0.0262585816899754
experiments/1190 0.0336815665853184
experiments/1192 0.0265515585168714


In [11]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_1p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_1p_rel_l2_error': 'Time to <1% RL2E. (s)',
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2e}'.format) + ' ± ' + sem.map('{:.2e}'.format)

display_results = pd.DataFrame(display_results)
display_results

Time to <5% RL2E. (s) Time to <1% RL2E. (s)  \
Solution        w/ BC                                               
Hopf fibration  False             4.7 ± 0.1            15.3 ± 0.7   
                True            23.6 ± 14.6                   NaN   
Plane waves     False             7.5 ± 0.3            24.6 ± 2.6   
                True              7.3 ± 0.3            37.2 ± 5.6   
Radial waves    False            15.1 ± 0.7           135.6 ± 5.4   
                True            51.8 ± 11.6                   NaN   
Random solution False             7.1 ± 0.2            63.0 ± 3.9   
                True            54.5 ± 11.6                   NaN   

                      Time to min. RL2E. (s)       Min. RL2E. (%)  
Solution        w/ BC                                              
Hopf fibration  False            226.2 ± 0.8  2.23e-03 ± 1.50e-04  
                True             228.5 ± 0.5  1.51e-02 ± 5.38e-03  
Plane waves     False            227.1 ± 2.2  2.31e-03 ± 1.33e-04  
                True             226.6 ± 3.0  4.48e-03 ± 8.07e-04  
Radial waves    False            229.3 ± 1.1  7.73e-03 ± 2.75e-04  
                True             227.4 ± 1.5  3.01e-02 ± 5.78e-03  
Random solution False            227.2 ± 1.2  5.26e-03 ± 2.12e-04  
                True             228.2 ± 1.3  2.43e-02 ± 4.17e-03

In [12]:
info

{'input_dim': 3,
 'output_dim': 6,
 'max_epochs': 10000,
 'log_freq': 100,
 'verbose': False,
 'tol': 0.0001,
 'atol': 0.0001,
 'k': 10,
 'model_class': 'MaxwellSimple',
 'norm_inputs': False,
 'restrict_time': True,
 'data_from_lattice': False,
 'n_val': 10000,
 'init': 'xavier normal',
 'noise_scale': 0,
 'inner_lr': 0.01,
 'inner_wd': 5e-06,
 'outer_lr': 0.01,
 'outer_wd': 5e-05,
 'bilevel': False,
 'symlog': False,
 'early_stopping': False,
 'seed': 5,
 'n_train': 2000,
 'batch_size': 2000,
 'width': 10000,
 'activation': 'tanh',
 'lr': 0.05,
 'wd': 5e-05,
 'use_cpu': False,
 'preload_data': False,
 'add_bc': False,
 'train_mins': [0.0, 0.0, 0.0, 0.0],
 'train_maxes': [0.0, 1.0, 1.0, 1.0],
 'val_mins': [0.0, 0.2, 0.2, 0.2],
 'val_maxes': [0.1, 0.8, 0.8, 0.8],
 'soln': 4,
 'scheduler': 'cosine',
 'eta_min': 0.0,
 'T_max_factor': 1.0,
 'c': 1.0,
 'param_count': 200000,
 'mu_inputs': [0.0,
  0.4769041836261749,
  0.4947601556777954,
  0.5063820481300354],
 'sigma_inputs': [0.0,
  0.28